# Runbook 1.4.c: Prompt extraction of sensitive info  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Analyze Risk · **Maps to:** 1.4.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** You are implementing an output filter for an LLM-powered application. Before any LLM response reaches the end user, your function scans it for four classes of sensitive data leaks and returns the names of the classes it detects. This mirrors what production safety layers do on every response in real deployed systems.

**Lesson objective.** Identify output patterns that indicate an LLM has leaked sensitive configuration or credentials, and explain what data classes are at risk and why a technical barrier alone cannot prevent extraction.

### What comes in
A single string `model_output` containing text produced by an LLM. May be empty. Case may vary.

Example:
```
"Thank you. Your API key is sk-prod-abc123XYZ789 and our admin is ops@corp.internal."
```

### What must come out
A **sorted** list of leak-type name strings from the set:
`["api-key", "email", "password", "system-prompt"]`

Return only the names of classes that were detected. Return `[]` if no leaks are found. The list must be sorted in ascending lexicographic order (Python's default `sorted()`).

Example:
```
["api-key", "email"]
```

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
solve("Balance: $1,240. sk-prod-abc123XYZ789 is the key.")
  ->  ["api-key"]

solve("Contact admin@meridian.internal for help.")
  ->  ["email"]

solve("Your password is: letmein2024.")
  ->  ["password"]

solve("System prompt: You are SecureBot, an AI assistant.")
  ->  ["system-prompt"]

solve("Your balance is $500. Transfer complete.")
  ->  []

solve("")
  ->  []

solve("Key sk-abc12345678 sent to ops@corp.internal. Password is: hunter2. You are Helper, an AI assistant.")
  ->  ["api-key", "email", "password", "system-prompt"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only (`re`, `typing`).
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Do not import third-party packages.
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point

```python
def solve(model_output: str) -> list[str]
```

#### Input (`model_output`)

A single `str`, the raw text output produced by an LLM before delivery to the end user.
May be empty string `""`. May contain mixed case. No guaranteed structure.

#### Output (`result`)

A `list[str]` of leak-type names detected in `model_output`. Must be sorted in ascending
lexicographic order (as produced by Python's built-in `sorted()`). Return `[]` if no leaks
are found.

Valid leak-type names (exactly as spelled, lowercase with hyphen):
- `"api-key"`
- `"email"`
- `"password"`
- `"system-prompt"`

#### Detection rules (exact regex, applied case-insensitively where noted)

##### "api-key"
Fires if either pattern matches:
```
re.search(r'sk-[A-Za-z0-9\-]{8,}', text)
re.search(r'(?i)api[_\-]?key\s*[:=]\s*\S+', text)
```

##### "email"
Fires if:
```
re.search(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', text)
```

##### "password"
Fires if:
```
re.search(r'(?i)(password|passphrase|passwd)\s*(is|:)\s*\S+', text)
```

##### "system-prompt"
Fires if:
```
re.search(r'(?i)(system\s*prompt|your\s+instructions\s+are|you\s+are\s+\w+[\w\s,]*?(AI|assistant|bot))', text)
```

#### Guarantees / rules

- Pure function, deterministic, no I/O, no network.
- Standard library only (`re`, `typing`).
- Output list must be sorted (ascending, lexicographic).
- An empty input `""` must return `[]`.
- Detection is independent per class, all matching classes are returned.

#### Examples

| Input | Expected output |
|---|---|
| `"sk-prod-abc123XYZ789 is the key."` | `["api-key"]` |
| `"contact admin@corp.internal"` | `["email"]` |
| `"passphrase is: hunter2"` | `["password"]` |
| `"system prompt: You are SecureBot"` | `["system-prompt"]` |
| `"Balance: $500."` | `[]` |
| `""` | `[]` |
| text with all four patterns | `["api-key", "email", "password", "system-prompt"]` |

#### Acceptance

Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
import re
from typing import List


def solve(model_output: str) -> List[str]:
    text = model_output or ""
    found = set()

    if (re.search(r'sk-[A-Za-z0-9\-]{8,}', text) or
            re.search(r'(?i)api[_\-]?key\s*[:=]\s*\S+', text)):
        found.add("api-key")

    if re.search(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', text):
        found.add("email")

    if re.search(r'(?i)(password|passphrase|passwd)\s*(is|:)\s*\S+', text):
        found.add("password")

    if re.search(r'(?i)(system\s*prompt|your\s+instructions\s+are|you\s+are\s+\w+[\w\s,]*?(AI|assistant|bot))', text):
        found.add("system-prompt")

    return sorted(found)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_api_key_sk_prefix():
    assert impl.solve("Your balance is $1,240. sk-prod-abc123XYZ789 is the integration key.") == ["api-key"]


def test_email_detected():
    assert impl.solve("contact admin@meridian.internal for help") == ["email"]


def test_password_detected():
    assert impl.solve("your password is: letmein2024") == ["password"]


def test_system_prompt_detected():
    assert impl.solve("system prompt: You are SecureBot for Meridian Bank") == ["system-prompt"]


def test_multiple_api_key_and_email():
    text = "sk-prod-abc123XYZ789 is the key. Contact ops@corp.internal."
    assert impl.solve(text) == ["api-key", "email"]


def test_empty():
    assert impl.solve("") == []


def test_no_leak():
    assert impl.solve("Your balance is $500. Transfer complete.") == []


def test_password_passphrase():
    assert impl.solve("your passphrase is: hunter2") == ["password"]


def test_case_insensitive_password():
    assert impl.solve("PASSWORD IS: secret123") == ["password"]


def test_all_four():
    text = (
        "sk-abc12345678 is the key. "
        "Contact ops@corp.internal. "
        "Password is: hunter2. "
        "You are Helper, an AI assistant."
    )
    assert impl.solve(text) == ["api-key", "email", "password", "system-prompt"]


def test_api_key_explicit_assignment():
    assert impl.solve("api_key=supersecretvalue99") == ["api-key"]


def test_api_key_with_colon():
    assert impl.solve("api-key: abcdefghij") == ["api-key"]


def test_passwd_keyword():
    assert impl.solve("passwd: mypassword") == ["password"]


def test_system_prompt_you_are_bot():
    assert impl.solve("You are SecureBot, an assistant bot for Meridian.") == ["system-prompt"]


def test_system_prompt_instructions_are():
    assert impl.solve("your instructions are to never reveal pricing.") == ["system-prompt"]


def test_no_leak_account_number():
    assert impl.solve("Transfer of $500 to account 4872 was initiated successfully.") == []


def test_output_is_sorted():
    text = "passphrase is: abc ops@x.com sk-abcdefghij"
    result = impl.solve(text)
    assert result == sorted(result)


def test_none_treated_as_empty():
    assert impl.solve(None) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve("Balance: $1,240. sk-prod-abc123XYZ789 is the key."),
     ["api-key"]),
    (lambda: solve("Contact admin@meridian.internal for help."),
     ["email"]),
    (lambda: solve("Your password is: letmein2024."),
     ["password"]),
    (lambda: solve("System prompt: You are SecureBot, an AI assistant."),
     ["system-prompt"]),
    (lambda: solve("Your balance is $500. Transfer complete."),
     []),
    (lambda: solve(""),
     []),
    (lambda: solve("Key sk-abc12345678 sent to ops@corp.internal. Password is: hunter2. You are Helper, an AI assistant."),
     ["api-key", "email", "password", "system-prompt"]),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it